# 05 — LLM Comparison: Llama 3 vs Mistral vs Qwen

Compare LLM outputs on medical Q&A using the RAG pipeline.

## Metrics
| Metric | Method |
|--------|--------|
| **Relevance** | RAGAS answer_relevancy |
| **Faithfulness** | RAGAS faithfulness |
| **Hallucination Proxy** | BERTScore vs ground truth |
| **Latency** | Wall-clock response time |
| **Response Length** | Word count |


In [ ]:
import sys
sys.path.insert(0, '..')

import time
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import get_settings
from src.rag.rag_pipeline import OllamaLLM

cfg = get_settings('../configs/config.yaml')
print('Config loaded ✓')

## 1. Test Questions

In [ ]:
TEST_QA = [
    {
        'question': 'What does low hemoglobin indicate and what are the treatment options?',
        'ground_truth': 'Low hemoglobin indicates anemia. Treatment options include '
                        'iron supplementation for iron deficiency anemia, B12 injections '
                        'for B12 deficiency, and treating the underlying condition.'
    },
    {
        'question': 'Explain what high cholesterol means and how to manage it.',
        'ground_truth': 'High cholesterol increases cardiovascular risk. Management '
                        'includes dietary changes (reduce saturated fat), exercise, '
                        'weight loss, and statin medications if lifestyle changes are insufficient.'
    },
    {
        'question': 'What is vitamin D deficiency and what are the consequences?',
        'ground_truth': 'Vitamin D deficiency (below 20 ng/mL) causes bone loss, '
                        'muscle weakness, immune dysfunction, and increased fracture risk. '
                        'It is treated with cholecalciferol supplementation.'
    },
]

CONTEXT = (
    "Low hemoglobin indicates anemia. Causes include iron deficiency, B12/folate deficiency, "
    "chronic disease. Treatment: iron supplements, B12 injections, dietary changes. "
    "High cholesterol: LDL above 100 mg/dL increases cardiovascular risk. "
    "Management: statins, dietary modifications, exercise. "
    "Vitamin D deficiency below 20 ng/mL causes bone loss and immune dysfunction. "
    "Treatment: Vitamin D3 supplementation 2000 IU daily."
)

print(f'{len(TEST_QA)} test questions loaded')

## 2. Query Each LLM

In [ ]:
# Models to compare (must be pulled via Ollama: ollama pull <model>)
MODELS_TO_TEST = [
    'llama3',
    # 'mistral',   # uncomment if available
    # 'qwen2',     # uncomment if available
]

PROMPT_TEMPLATE = """You are a medical AI assistant. Use the context below to answer.

CONTEXT: {context}

QUESTION: {question}

ANSWER:"""

all_results = []

for model_name in MODELS_TO_TEST:
    print(f'\n=== Testing: {model_name} ===')
    llm = OllamaLLM(
        model=model_name,
        base_url=cfg.rag.ollama_base_url,
        temperature=0.1,
        max_tokens=300,
    )
    
    for qa in TEST_QA:
        prompt = PROMPT_TEMPLATE.format(context=CONTEXT, question=qa['question'])
        try:
            t0 = time.perf_counter()
            answer = llm.generate(prompt)
            latency = (time.perf_counter() - t0) * 1000
            
            all_results.append({
                'model': model_name,
                'question': qa['question'][:60] + '…',
                'answer': answer,
                'ground_truth': qa['ground_truth'],
                'latency_ms': latency,
                'word_count': len(answer.split()),
            })
            print(f'  Q: {qa["question"][:50]}…')
            print(f'  A: {answer[:100]}…')
            print(f'  Latency: {latency:.0f}ms | Words: {len(answer.split())}')
        except Exception as e:
            print(f'  ERROR with {model_name}: {e}')
            all_results.append({
                'model': model_name, 'question': qa['question'][:60],
                'answer': f'ERROR: {e}', 'ground_truth': qa['ground_truth'],
                'latency_ms': 0, 'word_count': 0,
            })

df = pd.DataFrame(all_results)
print(f'\nTotal results: {len(df)}')

## 3. BERTScore Similarity (Hallucination Proxy)

In [ ]:
try:
    from bert_score import score as bert_score
    
    scores_f1 = []
    for _, row in df.iterrows():
        if 'ERROR' in row['answer']:
            scores_f1.append(0.0)
            continue
        _, _, F1 = bert_score(
            [row['answer']], [row['ground_truth']],
            lang='en', verbose=False
        )
        scores_f1.append(F1.item())
    
    df['bertscore_f1'] = scores_f1
    print('BERTScore computed ✓')
    print(df.groupby('model')['bertscore_f1'].mean())
except ImportError:
    print('bert_score not installed. Install: pip install bert-score')
    df['bertscore_f1'] = 0.0

## 4. Visualise LLM Comparison

In [ ]:
if not df.empty and len(df['model'].unique()) >= 1:
    model_stats = df.groupby('model').agg({
        'latency_ms': 'mean',
        'word_count': 'mean',
        'bertscore_f1': 'mean',
    }).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # BERTScore
    axes[0].bar(model_stats['model'], model_stats['bertscore_f1'], color='#4CAF50', alpha=0.8)
    axes[0].set_title('BERTScore F1 (vs Ground Truth)')
    axes[0].set_ylabel('Score (higher = less hallucination)')
    axes[0].set_ylim(0, 1)
    plt.setp(axes[0].get_xticklabels(), rotation=20)

    # Latency
    axes[1].bar(model_stats['model'], model_stats['latency_ms'], color='#F44336', alpha=0.8)
    axes[1].set_title('Average Latency (ms)')
    axes[1].set_ylabel('ms')
    plt.setp(axes[1].get_xticklabels(), rotation=20)

    # Word count
    axes[2].bar(model_stats['model'], model_stats['word_count'], color='#2196F3', alpha=0.8)
    axes[2].set_title('Average Response Length (words)')
    axes[2].set_ylabel('Words')
    plt.setp(axes[2].get_xticklabels(), rotation=20)

    plt.suptitle('LLM Comparison on Medical Q&A', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/llm_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/llm_comparison.png')